In [ ]:
from google.colab import drive
import pandas as pd
import torch
from tqdm import tqdm
import numpy as np
import glob
import re
from collections import defaultdict
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, GPTNeoXForCausalLM

In [ ]:
drive.mount('/content/drive')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2", padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.config.pad_token_id = model.config.eos_token_id

In [ ]:
def to_tokens_and_logprobs(model, tokenizer, input_texts):
    # pad the sentence
    bos_token = tokenizer.bos_token if tokenizer.bos_token is not None else tokenizer.pad_token
    padded_texts = [bos_token + " " + text for text in input_texts]

    input_ids = tokenizer(padded_texts, padding=True, return_tensors="pt").input_ids
    outputs = model(input_ids)
    # probs = torch.log_softmax(outputs.logits, dim=-1).detach() # natural log
    probs = torch.softmax(outputs.logits, dim=-1).detach()
    surprisals = -1 * torch.log2(probs) # log base 2
    entropy = -torch.sum(probs * torch.log2(probs), dim=-1) # log base 2

    surprisals = surprisals[:, :-1, :]
    probs = probs[:, :-1, :]
    input_ids = input_ids[:, 1:]
    entropy = entropy[:, :-1]

    # collect the surprisal of the generated token -- probability at index 0 corresponds to the token at index 1
    gen_surprisals = torch.gather(surprisals, 2, input_ids[:, :, None]).squeeze(-1)

    # collect the log probabilities of the generated token -- probability at index 0 corresponds to the token at index 1
    gen_probs = torch.gather(probs, 2, input_ids[:, :, None]).squeeze(-1)

    text_sequence = []
    for input_sentence, input_surp, input_probs, ent_seq in zip(input_ids, gen_surprisals, gen_probs, entropy):
        for token, s, p, e in zip(input_sentence, input_surp, input_probs, ent_seq):
            if token not in tokenizer.all_special_ids:
                text_sequence.append((tokenizer.decode(token), s.item(), p.item(), e.item()))
    return text_sequence

In [ ]:
sentences = pd.concat([pd.read_csv("/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_1_processed_1_noquote.csv")],ignore_index=True)

In [ ]:
len(sentences)

In [ ]:
for i, row in tqdm(sentences.iterrows()):
  sentence = row.sentence

  # skip if there is no embedded clause and the embedded clause is of other type
  if pd.isna(row["complement_type"]):
    continue
  # skip if the embedded clause precedes the matrix verb, likely a quotation
  if row["matrix_predicate_to_cc"] < 0:
    continue

  matrix_verb_id = row.matrix_predicate_id
  matrix_verb_to_cc = row.matrix_predicate_to_cc
  context_verb_no_that = row.one_word_omit_that
  context_verb_that = row.one_word_with_that
  current_context_verb_no_that = row.one_word_current_omit_that
  current_context_verb_that = row.one_word_current_with_that

  # create the context+verb+"that" from context+verb+"that"+w1
  context_verb_only_that = context_verb_that.split()
  context_verb_only_that = context_verb_only_that[:-1]
  context_verb_only_that = " ".join(context_verb_only_that)
  current_context_verb_only_that = current_context_verb_that.split()
  current_context_verb_only_that = current_context_verb_only_that[:-1]
  current_context_verb_only_that = " ".join(current_context_verb_only_that)

  # 1. get the surprisal and entropy of the verb
  context_verb = row.matrix_span_verb
  if pd.isna(row["matrix_span_no_verb"]):
    continue
  else:
    context = row.matrix_span_no_verb

  logprobs_context = to_tokens_and_logprobs(model, tokenizer, [context])
  length_context = len(logprobs_context)

  logprobs_context_verb = to_tokens_and_logprobs(model, tokenizer, [context_verb])
  length_context_verb = len(logprobs_context_verb)

  length_verb = length_context_verb-length_context

  informativity_verb_sum = 0
  for n in range(length_context, length_context_verb): # in case it is multiple tokens
    informativity = logprobs_context_verb[n][1]
    informativity_verb_sum += informativity
  informativity_verb_sum = informativity_verb_sum
  sentences.loc[i,"verb_sum"] = informativity_verb_sum # should NOT be divided by the number of tokens (length_context_verb-length_context)

  # 2. get the surprsial and entropy of first word in the complementizer clause
  # from the counterfactual sentence with no that
  logprobs_context_verb_no_that = to_tokens_and_logprobs(model, tokenizer, [context_verb_no_that])
  length_context_verb_no_that = len(logprobs_context_verb_no_that)

  length_w_no_that = length_context_verb_no_that - length_context_verb

  informativity_verb_no_that_sum = 0
  for n in range(length_context_verb, length_context_verb_no_that):
    informativity = logprobs_context_verb_no_that[n][1]
    informativity_verb_no_that_sum += informativity
  informativity_verb_no_that_sum = informativity_verb_no_that_sum
  sentences.loc[i,"cc_no_that_sum"] = informativity_verb_no_that_sum

  # from the counterfactual sentence with that
  logprobs_context_verb_only_that = to_tokens_and_logprobs(model, tokenizer, [context_verb_only_that])
  length_context_verb_only_that = len(logprobs_context_verb_only_that)

  logprobs_context_verb_that = to_tokens_and_logprobs(model, tokenizer, [context_verb_that])
  length_context_verb_that = len(logprobs_context_verb_that)

  length_w_that = length_context_verb_that - length_context_verb_only_that

  informativity_verb_that_sum = 0
  for n in range(length_context_verb_only_that, length_context_verb_that):
    informativity = logprobs_context_verb_that[n][1]
    informativity_verb_that_sum += informativity
  informativity_verb_that_sum = informativity_verb_that_sum
  sentences.loc[i,"cc_with_that_sum"] = informativity_verb_that_sum
  sentences.loc[i,"cc_n_sum"] = informativity_verb_no_that_sum + informativity_verb_that_sum

  # 3. get the surprsial and entropy of verb in the complementizer clause in the current context
  current_context_verb = row.matrix_span_current_verb
  if pd.isna(row["matrix_span_current_no_verb"]):
    continue
  else:
    current_context = row.matrix_span_current_no_verb

  current_informativity_verb_sum = 0
  if current_context == context and current_context_verb == context_verb:
    current_informativity_verb_sum = informativity_verb_sum
  else:
    logprobs_current_context = to_tokens_and_logprobs(model, tokenizer, [current_context])
    length_current_context = len(logprobs_current_context)

    logprobs_current_context_verb = to_tokens_and_logprobs(model, tokenizer, [current_context_verb])
    length_current_context_verb = len(logprobs_current_context_verb)

    length_current_verb = length_current_context_verb - length_current_context

    for n in range(length_current_context, length_current_context_verb):
      informativity = logprobs_current_context_verb[n][1]
      current_informativity_verb_sum += informativity
    current_informativity_verb_sum = current_informativity_verb_sum
  sentences.loc[i,"current_verb_sum"] = current_informativity_verb_sum

  # 4. get the surprsial and entropy of first word in the complementizer clause in the current context
  # from the counterfactual sentence with no that
  current_informativity_verb_no_that_sum = 0
  if current_context_verb_no_that == context_verb_no_that:
    current_informativity_verb_no_that_sum = informativity_verb_no_that_sum
  else:
    logprobs_current_context_verb_no_that = to_tokens_and_logprobs(model, tokenizer, [current_context_verb_no_that])
    length_current_context_verb_no_that = len(logprobs_current_context_verb_no_that)

    length_current_w_no_that = length_current_context_verb_no_that - length_current_context_verb

    for n in range(length_current_context_verb, length_current_context_verb_no_that):
      informativity = logprobs_current_context_verb_no_that[n][1]
      current_informativity_verb_no_that_sum += informativity
    current_informativity_verb_no_that_sum = current_informativity_verb_no_that_sum
  sentences.loc[i,"current_cc_no_that_sum"] = current_informativity_verb_no_that_sum

  # from the counterfactual sentence with that
  current_informativity_verb_that_sum = 0
  if current_context_verb_that == context_verb_that:
    current_informativity_verb_that_sum = informativity_verb_that_sum
  else:
    logprobs_currrent_context_verb_only_that = to_tokens_and_logprobs(model, tokenizer, [current_context_verb_only_that])
    length_current_context_verb_only_that = len(logprobs_currrent_context_verb_only_that)

    logprobs_current_context_verb_that = to_tokens_and_logprobs(model, tokenizer, [current_context_verb_that])
    length_current_context_verb_that = len(logprobs_current_context_verb_that)

    length_current_w_that = length_current_context_verb_that - length_current_context_verb_only_that

    for n in range(length_current_context_verb_only_that, length_current_context_verb_that):
      informativity = logprobs_current_context_verb_that[n][1]
      current_informativity_verb_that_sum += informativity
    current_informativity_verb_that_sum = current_informativity_verb_that_sum
  sentences.loc[i,"current_cc_with_that_sum"] = current_informativity_verb_that_sum
  sentences.loc[i,"current_cc_n_sum"] = current_informativity_verb_no_that_sum + current_informativity_verb_that_sum

sentences.to_csv("/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_1_noquote_surprisal.csv", index=False)